# 14 — Build Country-Year Feature Matrix

Assembles the master **country-year panel** used by notebooks 15–17 for instability prediction.
Reads cleaned parquets written by notebooks 01–13 from ADLS, joins them on a common ISO3 key,
and codes five binary outcome labels (all forward-shifted one year).

## Panel dimensions
- **Unit:** country-year (~167 countries × 25 years = ~4,000 rows)
- **Feature window:** 2000–2024
- **Training labels available for:** 2000–2023 (label at year *t* requires data from year *t+1*)

## Outcome labels (binary, forward-shifted +1 year)

| Label | Ground truth | Coding rule |
|---|---|---|
| `civil_war_onset` | UCDP-GED | First year of state-based conflict after ≥2-year peace spell |
| `coup_attempt` | Powell-Thyne | Any coup attempt (success or failure) in year *t+1* |
| `regime_backsliding` | V-Dem | `v2x_libdem` drops ≥0.05 in year *t+1*, or regime transitions to closed autocracy |
| `mass_unrest_onset` | ACLED | Annual protest+riot events exceed country 90th percentile in year *t+1* |
| `humanitarian_crisis_onset` | FEWS NET | Country enters IPC Phase ≥4 in year *t+1*, given Phase ≤3 at *t* (~39 countries) |

## Sources joined
Monthly sources (ACLED, GDELT, FAO food prices) are **aggregated to annual statistics**
before joining — the panel unit is country-year throughout.

## ADLS output
```
processed/feature_matrix/{RUN_DATE}/feature_matrix.parquet   — full feature panel
processed/feature_matrix/{RUN_DATE}/labels.parquet           — iso3 + year + 5 outcome columns
```

## Required environment variables
```
ADLS_ACCOUNT_NAME
ADLS_CONTAINER  (default: 'data')
```

In [ ]:
import os
import warnings
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from azure.identity import DefaultAzureCredential
import adlfs

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 40)

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

PANEL_START_YEAR  = 2000
PANEL_END_YEAR    = 2024
LABEL_HORIZON     = 1   # predict outcomes 1 year ahead

# Temporal split boundaries (year-inclusive)
TRAIN_END_YEAR    = 2018
VAL_END_YEAR      = 2021
# Test: 2022–2024

# ADLS prefixes for each raw source (latest date partition selected automatically)
RAW_PREFIXES = {
    "acled_monthly":    "raw/acled/monthly_agg",
    "wdi":              "raw/world_bank/wdi",
    "wgi":              "raw/world_bank/wgi",
    "vdem":             "raw/vdem",
    "polity5":          "raw/polity5",
    "ucdp_ged_cy":      "raw/ucdp_ged",
    "powell_thyne":     "raw/powell_thyne",
    "pitf":             "raw/pitf",
    "fsi":              "raw/fsi",
    "unhcr":            "raw/unhcr",
    "undp_hdi":         "raw/undp_hdi",
    "gdelt":            "raw/gdelt",
    "fao_ffpi":         "raw/fao/ffpi",
    "fao_cpi":          "raw/fao/country_cpi",
    "sipri":            "raw/sipri",
    "prio_grid_cy":     "raw/prio_grid",
    "archigos_cy":      "raw/archigos",
    "alc_cy":           "raw/alc",
    "cnts":             "raw/cnts",
    "nelda":            "raw/nelda",
}

print(f"Run date       : {RUN_DATE}")
print(f"Panel years    : {PANEL_START_YEAR}–{PANEL_END_YEAR}")
print(f"Temporal split : train ≤{TRAIN_END_YEAR} | val ≤{VAL_END_YEAR} | test >={VAL_END_YEAR+1}")

## ADLS helpers

In [ ]:
credential     = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential":   credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

def read_latest_parquet(prefix: str) -> pd.DataFrame | None:
    """
    List all date-partitioned subdirectories under `prefix` and read
    the parquet from the lexicographically latest one (most recent run date).
    Returns None if no parquet files are found.
    """
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    full_prefix = f"{ADLS_CONTAINER}/{prefix}"
    try:
        entries = fs.ls(full_prefix, detail=False)
    except FileNotFoundError:
        print(f"  WARNING: prefix not found: {full_prefix}")
        return None

    # Keep only date-partition directories (8-digit names)
    date_dirs = sorted(
        [e for e in entries if re.search(r'/\d{8}(/|$)', e)],
        reverse=True,
    )
    if not date_dirs:
        print(f"  WARNING: no date partitions found under {full_prefix}")
        return None

    latest_dir = date_dirs[0]
    parquet_files = [
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/"
        + f.replace(f"{ADLS_CONTAINER}/", "", 1)
        for f in fs.glob(f"{latest_dir}/*.parquet")
    ]
    if not parquet_files:
        print(f"  WARNING: no .parquet files in {latest_dir}")
        return None

    dfs = [pd.read_parquet(p, storage_options=storage_options) for p in parquet_files]
    df = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]
    print(f"  Loaded {len(df):,} rows from {latest_dir}")
    return df

## Country crosswalk

Loads the static crosswalk from `data/country_crosswalk.csv` (committed to the repo).
Provides lookup dicts for each join-key type → ISO3.

In [ ]:
# The crosswalk CSV lives alongside the notebooks in the repo
_crosswalk_path = Path("../data/country_crosswalk.csv")
if not _crosswalk_path.exists():
    _crosswalk_path = Path("data/country_crosswalk.csv")

df_cw = pd.read_csv(_crosswalk_path, dtype=str)
df_cw["cow_numeric"]  = pd.to_numeric(df_cw["cow_numeric"],  errors="coerce")
df_cw["gw_numeric"]   = pd.to_numeric(df_cw["gw_numeric"],   errors="coerce")
df_cw["iso_numeric"]  = pd.to_numeric(df_cw["iso_numeric"],  errors="coerce")
df_cw["fews_monitored"] = df_cw["fews_monitored"].astype(int)

# Lookup dicts  →  ISO3
cow_to_iso3  = dict(zip(df_cw["cow_numeric"],  df_cw["iso3"]))
gw_to_iso3   = dict(zip(df_cw["gw_numeric"],   df_cw["iso3"]))
iso2_to_iso3 = dict(zip(df_cw["iso2"],         df_cw["iso3"]))
inum_to_iso3 = dict(zip(df_cw["iso_numeric"],  df_cw["iso3"]))
cameo_to_iso3 = dict(zip(df_cw["cameo2"],      df_cw["iso3"]))

# Name normalisation helper for string-based sources (SIPRI, FSI, UNDP)
_NAME_OVERRIDES = {
    "cote d'ivoire":              "CIV",
    "ivory coast":                "CIV",
    "iran, islamic rep.":         "IRN",
    "iran (islamic republic of)": "IRN",
    "republic of korea":          "KOR",
    "korea, rep.":                "KOR",
    "korea, south":               "KOR",
    "democratic republic of the congo": "COD",
    "congo, dem. rep.":           "COD",
    "dr congo":                   "COD",
    "congo, rep.":                "COG",
    "syrian arab republic":       "SYR",
    "bolivia (plurinational state of)": "BOL",
    "tanzania, united republic of":    "TZA",
    "united republic of tanzania":     "TZA",
    "viet nam":                   "VNM",
    "lao pdr":                    "LAO",
    "lao people's democratic republic": "LAO",
    "kyrgyz republic":            "KGZ",
    "czech republic":             "CZE",
    "slovak republic":            "SVK",
    "turkiye":                    "TUR",
    "turkey":                     "TUR",
    "egypt, arab rep.":           "EGY",
    "gambia, the":                "GMB",
    "bahamas, the":               "BHS",
    "yemen, rep.":                "YEM",
    "venezuela, rb":              "VEN",
    "micronesia, fed. sts.":      "FSM",
    "russian federation":         "RUS",
    "timor-leste":                "TLS",
    "east timor":                 "TLS",
    "cabo verde":                 "CPV",
    "eswatini":                   "SWZ",
    "swaziland":                  "SWZ",
    "north macedonia":            "MKD",
    "macedonia, former yugoslav republic of": "MKD",
}
_name_to_iso3 = dict(zip(
    df_cw["country_name_canonical"].str.lower().str.strip(),
    df_cw["iso3"]
))

def name_to_iso3(name: str) -> str | None:
    if not isinstance(name, str):
        return None
    key = re.sub(r"[^a-z0-9 ]", "", name.lower().strip())
    key_punc = name.lower().strip()
    return (
        _NAME_OVERRIDES.get(key_punc)
        or _name_to_iso3.get(key_punc)
        or _name_to_iso3.get(key)
    )

FEWS_COUNTRIES = set(df_cw.loc[df_cw["fews_monitored"] == 1, "iso3"])

print(f"Crosswalk loaded: {len(df_cw)} countries")
print(f"FEWS-monitored  : {len(FEWS_COUNTRIES)} countries")

## Panel spine

Create the rectangular country-year grid: all ISO3 codes × all years 2000–2024.
Every source is left-joined onto this spine so that absence of data produces NaN
(not a dropped row).

In [ ]:
years    = list(range(PANEL_START_YEAR, PANEL_END_YEAR + 1))
iso3s    = df_cw["iso3"].tolist()

df_spine = pd.DataFrame(
    [(iso3, yr) for iso3 in iso3s for yr in years],
    columns=["iso3", "year"],
)

print(f"Panel spine: {len(df_spine):,} country-year rows "
      f"({len(iso3s)} countries × {len(years)} years)")

## Load and join annual sources

Each annual source is read, its country identifier mapped to ISO3 via the crosswalk,
then left-joined onto the spine on `(iso3, year)`.

Sources: World Bank WDI, WGI, V-Dem, Polity5, UCDP country-year, FSI, SIPRI,
PRIO-GRID country-year aggregates, Archigos, ALC, CNTS, NELDA, UNHCR, UNDP HDI.

In [ ]:
def _add_iso3_from_cow(df, cow_col="ccode"):
    df = df.copy()
    df["iso3"] = df[cow_col].map(cow_to_iso3)
    return df

def _add_iso3_from_gw(df, gw_col="gwno"):
    df = df.copy()
    df["iso3"] = df[gw_col].map(gw_to_iso3)
    return df

def _add_iso3_from_name(df, name_col):
    df = df.copy()
    df["iso3"] = df[name_col].apply(name_to_iso3)
    return df

def _log_join(source_name, df_raw, df_joined):
    n_raw   = df_raw["iso3"].notna().sum()
    n_match = df_joined["iso3"].notna().sum()
    unmatched = df_raw[df_raw["iso3"].isna()].shape[0]
    if unmatched:
        print(f"  {source_name}: {unmatched} unmatched rows ({unmatched/len(df_raw):.1%})")

panel = df_spine.copy()

# ── World Bank WDI ────────────────────────────────────────────────────────────
raw = read_latest_parquet(RAW_PREFIXES["wdi"])
if raw is not None:
    raw.columns = [c.lower().strip() for c in raw.columns]
    # wbgapi output uses 'economy' (ISO3) and 'time' (year integer)
    eco_col  = next((c for c in raw.columns if c in ("economy", "iso3", "country_code")), None)
    yr_col   = next((c for c in raw.columns if c in ("time", "year")), None)
    if eco_col and yr_col:
        raw = raw.rename(columns={eco_col: "iso3", yr_col: "year"})
        raw["year"] = pd.to_numeric(raw["year"], errors="coerce").astype("Int64")
        drop = [c for c in raw.columns if c in ("country_name",)]
        wdi_cols = [c for c in raw.columns if c not in drop + ["iso3", "year"]]
        raw = raw[["iso3", "year"] + wdi_cols]
        panel = panel.merge(raw, on=["iso3", "year"], how="left")
        print(f"WDI: {len(wdi_cols)} feature columns added")

# ── World Bank WGI ────────────────────────────────────────────────────────────
raw = read_latest_parquet(RAW_PREFIXES["wgi"])
if raw is not None:
    raw.columns = [c.lower().strip() for c in raw.columns]
    eco_col = next((c for c in raw.columns if c in ("economy", "iso3", "country_code")), None)
    yr_col  = next((c for c in raw.columns if c in ("time", "year")), None)
    if eco_col and yr_col:
        raw = raw.rename(columns={eco_col: "iso3", yr_col: "year"})
        raw["year"] = pd.to_numeric(raw["year"], errors="coerce").astype("Int64")
        wgi_cols = [c for c in raw.columns if c not in ("iso3", "year", "country_name")]
        raw = raw[["iso3", "year"] + wgi_cols]
        panel = panel.merge(raw, on=["iso3", "year"], how="left", suffixes=("", "_wgi"))
        print(f"WGI: {len(wgi_cols)} feature columns added")

# ── V-Dem ─────────────────────────────────────────────────────────────────────
raw = read_latest_parquet(RAW_PREFIXES["vdem"])
if raw is not None:
    raw.columns = [c.lower().strip() for c in raw.columns]
    iso3_col = next((c for c in raw.columns if c in ("country_text_id", "iso3")), None)
    yr_col   = next((c for c in raw.columns if c in ("year",)), None)
    if iso3_col and yr_col:
        raw = raw.rename(columns={iso3_col: "iso3"})
        raw["year"] = pd.to_numeric(raw["year"], errors="coerce").astype("Int64")
        # Fall back to COW code for rows where iso3 not in crosswalk
        if "cowcode" in raw.columns:
            mask = ~raw["iso3"].isin(df_cw["iso3"])
            raw.loc[mask, "iso3"] = raw.loc[mask, "cowcode"].map(cow_to_iso3)
        vdem_cols = [c for c in raw.columns
                     if c not in ("iso3", "year", "cowcode", "country_name", "country_id")]
        raw = raw[["iso3", "year"] + vdem_cols]
        panel = panel.merge(raw, on=["iso3", "year"], how="left")
        print(f"V-Dem: {len(vdem_cols)} feature columns added")

# ── Polity5 ───────────────────────────────────────────────────────────────────
raw = read_latest_parquet(RAW_PREFIXES["polity5"])
if raw is not None:
    raw.columns = [c.lower().strip() for c in raw.columns]
    cow_col = next((c for c in raw.columns if c in ("ccode", "cow_code", "ccodecow")), None)
    yr_col  = next((c for c in raw.columns if c in ("year",)), None)
    if cow_col and yr_col:
        raw = _add_iso3_from_cow(raw, cow_col)
        raw["year"] = pd.to_numeric(raw["year"], errors="coerce").astype("Int64")
        polity_cols = [c for c in raw.columns
                       if c not in ("iso3", "year", cow_col, "country", "scode")]
        raw = raw[["iso3", "year"] + polity_cols]
        panel = panel.merge(raw, on=["iso3", "year"], how="left")
        print(f"Polity5: {len(polity_cols)} feature columns added")

print(f"\nPanel after annual sources: {panel.shape}")

In [ ]:
# ── Remaining annual sources ──────────────────────────────────────────────────

def _join_cow_source(prefix_key, feature_prefix, cow_col="ccode", yr_col="year",
                     drop_cols=()):
    raw = read_latest_parquet(RAW_PREFIXES[prefix_key])
    if raw is None:
        return
    raw.columns = [c.lower().strip() for c in raw.columns]
    ccol = next((c for c in raw.columns if c == cow_col or c in ("ccode","cow")), None)
    ycol = next((c for c in raw.columns if c in (yr_col, "year")), None)
    if not ccol or not ycol:
        print(f"  {prefix_key}: could not find COW/year columns")
        return
    raw = _add_iso3_from_cow(raw, ccol)
    raw["year"] = pd.to_numeric(raw[ycol], errors="coerce").astype("Int64")
    skip = set(drop_cols) | {ccol, ycol, "iso3", "country", "country_name",
                              "ccode", "idacr", "cow", "gwno"}
    feat_cols = [c for c in raw.columns if c not in skip]
    # Prefix feature columns to avoid collisions
    raw = raw.rename(columns={c: f"{feature_prefix}_{c}" for c in feat_cols})
    feat_cols_prefixed = [f"{feature_prefix}_{c}" for c in feat_cols]
    raw = raw[["iso3", "year"] + feat_cols_prefixed].drop_duplicates(["iso3","year"])
    global panel
    panel = panel.merge(raw, on=["iso3","year"], how="left")
    print(f"{prefix_key}: {len(feat_cols_prefixed)} columns added")

def _join_gw_source(prefix_key, feature_prefix, gw_col="gwno", yr_col="year",
                    drop_cols=()):
    raw = read_latest_parquet(RAW_PREFIXES[prefix_key])
    if raw is None:
        return
    raw.columns = [c.lower().strip() for c in raw.columns]
    gcol = next((c for c in raw.columns if c in (gw_col, "gwno", "gid")), None)
    ycol = next((c for c in raw.columns if c in (yr_col, "year")), None)
    if not gcol or not ycol:
        print(f"  {prefix_key}: could not find GW/year columns")
        return
    raw = _add_iso3_from_gw(raw, gcol)
    raw["year"] = pd.to_numeric(raw[ycol], errors="coerce").astype("Int64")
    skip = set(drop_cols) | {gcol, ycol, "iso3", "country", "country_name", "gwno"}
    feat_cols = [c for c in raw.columns if c not in skip]
    raw = raw.rename(columns={c: f"{feature_prefix}_{c}" for c in feat_cols})
    feat_cols_prefixed = [f"{feature_prefix}_{c}" for c in feat_cols]
    raw = raw[["iso3", "year"] + feat_cols_prefixed].drop_duplicates(["iso3","year"])
    global panel
    panel = panel.merge(raw, on=["iso3","year"], how="left")
    print(f"{prefix_key}: {len(feat_cols_prefixed)} columns added")

def _join_name_source(prefix_key, feature_prefix, name_col, yr_col="year",
                      drop_cols=()):
    raw = read_latest_parquet(RAW_PREFIXES[prefix_key])
    if raw is None:
        return
    raw.columns = [c.lower().strip() for c in raw.columns]
    ncol = next((c for c in raw.columns if c in (name_col, "country", "country_name")), None)
    ycol = next((c for c in raw.columns if c in (yr_col, "year")), None)
    if not ncol or not ycol:
        print(f"  {prefix_key}: could not find name/year columns")
        return
    raw = _add_iso3_from_name(raw, ncol)
    raw["year"] = pd.to_numeric(raw[ycol], errors="coerce").astype("Int64")
    skip = set(drop_cols) | {ncol, ycol, "iso3", "country", "country_name"}
    feat_cols = [c for c in raw.columns if c not in skip]
    raw = raw.rename(columns={c: f"{feature_prefix}_{c}" for c in feat_cols})
    feat_cols_prefixed = [f"{feature_prefix}_{c}" for c in feat_cols]
    raw = raw[["iso3", "year"] + feat_cols_prefixed].drop_duplicates(["iso3","year"])
    global panel
    panel = panel.merge(raw, on=["iso3","year"], how="left")
    print(f"{prefix_key}: {len(feat_cols_prefixed)} columns added")

# UCDP country-year aggregates (gwno)
_join_gw_source("ucdp_ged_cy",  "ucdp",     gw_col="gwno")
# Powell-Thyne coups (ccode/COW) — used later for labels; join summary cols now
_join_cow_source("powell_thyne","coup",      drop_cols=("date","day","month"))
# PITF (ccode)
_join_cow_source("pitf",        "pitf")
# FSI (country name)
_join_name_source("fsi",        "fsi",       name_col="country")
# SIPRI (country name)
_join_name_source("sipri",      "sipri",     name_col="country_name")
# PRIO-GRID country-year aggregates (gwno)
_join_gw_source("prio_grid_cy", "prio",      gw_col="gwno")
# Archigos (ccode)
_join_cow_source("archigos_cy", "arch",      drop_cols=("idacr",))
# ALC — Africa only; ccode
_join_cow_source("alc_cy",      "alc",       drop_cols=("country",))
# CNTS (ccode)
_join_cow_source("cnts",        "cnts",      drop_cols=("country",))
# NELDA (cow)
_join_cow_source("nelda",       "nelda",     cow_col="cow", drop_cols=("country",))
# UNHCR (iso3 direct)
raw = read_latest_parquet(RAW_PREFIXES["unhcr"])
if raw is not None:
    raw.columns = [c.lower().strip() for c in raw.columns]
    yr_col = next((c for c in raw.columns if c in ("year",)), None)
    iso_col = next((c for c in raw.columns if c in ("iso3","iso","country_of_asylum_iso")), None)
    if iso_col and yr_col:
        raw = raw.rename(columns={iso_col: "iso3"})
        raw["year"] = pd.to_numeric(raw[yr_col], errors="coerce").astype("Int64")
        feat_cols = [c for c in raw.columns if c not in ("iso3","year","country_name")]
        raw = raw.rename(columns={c: f"unhcr_{c}" for c in feat_cols})
        panel = panel.merge(raw[["iso3","year"]+[f"unhcr_{c}" for c in feat_cols]],
                            on=["iso3","year"], how="left")
        print(f"UNHCR: {len(feat_cols)} columns added")
# UNDP HDI (country name)
_join_name_source("undp_hdi",   "hdi",       name_col="country")
# FAO country CPI (country name or iso3)
_join_name_source("fao_cpi",    "fao_cpi",   name_col="country")

print(f"\nPanel after all annual sources: {panel.shape}")

## Aggregate monthly sources to annual and join

ACLED, GDELT, and FAO food price index are monthly. We compute annual summary
statistics (sum, mean, std) and join them onto the country-year panel.

In [ ]:
# ── ACLED monthly → annual ────────────────────────────────────────────────────
raw_acled = read_latest_parquet(RAW_PREFIXES["acled_monthly"])
if raw_acled is not None:
    raw_acled.columns = [c.lower().strip() for c in raw_acled.columns]

    # Resolve country identifier to ISO3
    iso_col = next((c for c in raw_acled.columns
                    if c in ("iso3", "iso", "country_iso")), None)
    if iso_col == "iso" and raw_acled["iso"].dtype in (int, float, "Int64"):
        # ISO numeric integer
        raw_acled["iso3"] = raw_acled["iso"].map(inum_to_iso3)
    elif iso_col:
        raw_acled = raw_acled.rename(columns={iso_col: "iso3"})
    else:
        name_col = next((c for c in raw_acled.columns if "country" in c), None)
        if name_col:
            raw_acled["iso3"] = raw_acled[name_col].apply(name_to_iso3)

    # Extract year from year_month ("YYYY-MM") or year column
    if "year" not in raw_acled.columns:
        ym_col = next((c for c in raw_acled.columns if "year_month" in c or c == "ym"), None)
        if ym_col:
            raw_acled["year"] = raw_acled[ym_col].str[:4].astype(int)

    count_cols = [c for c in raw_acled.columns
                  if c.startswith("events_") or c.startswith("fatalities")]

    if "iso3" in raw_acled.columns and "year" in raw_acled.columns and count_cols:
        acled_agg = (
            raw_acled[["iso3", "year"] + count_cols]
            .groupby(["iso3", "year"], as_index=False)
            .agg(
                **{f"acled_{c}_annual":    (c, "sum") for c in count_cols},
                **{f"acled_{c}_mean_mo":   (c, "mean") for c in count_cols},
                **{f"acled_{c}_std_mo":    (c, "std")  for c in count_cols},
            )
        )
        acled_agg["year"] = acled_agg["year"].astype("Int64")
        panel = panel.merge(acled_agg, on=["iso3", "year"], how="left")
        print(f"ACLED: {len(acled_agg.columns)-2} annual aggregate columns added")

# ── GDELT monthly → annual ────────────────────────────────────────────────────
raw_gdelt = read_latest_parquet(RAW_PREFIXES["gdelt"])
if raw_gdelt is not None:
    raw_gdelt.columns = [c.lower().strip() for c in raw_gdelt.columns]
    raw_gdelt["iso3"] = raw_gdelt["cameo_country"].map(cameo_to_iso3)

    if "year" not in raw_gdelt.columns:
        ym_col = next((c for c in raw_gdelt.columns if "year_month" in c), None)
        if ym_col:
            raw_gdelt["year"] = raw_gdelt[ym_col].str[:4].astype(int)

    gdelt_numeric = [c for c in raw_gdelt.columns
                     if c not in ("iso3", "year", "cameo_country", "year_month",
                                  "country_label", "cameo_country")
                     and raw_gdelt[c].dtype in ("float64", "int64", "Int64")]

    if "iso3" in raw_gdelt.columns and "year" in raw_gdelt.columns and gdelt_numeric:
        gdelt_agg_spec = {}
        for c in gdelt_numeric:
            if "count" in c or c.startswith("events_") or c == "mentions_total":
                gdelt_agg_spec[f"gdelt_{c}_annual"] = (c, "sum")
            else:
                gdelt_agg_spec[f"gdelt_{c}_mean"] = (c, "mean")
                gdelt_agg_spec[f"gdelt_{c}_std"]  = (c, "std")

        gdelt_agg = (
            raw_gdelt[["iso3", "year"] + gdelt_numeric]
            .groupby(["iso3", "year"], as_index=False)
            .agg(**gdelt_agg_spec)
        )
        gdelt_agg["year"] = gdelt_agg["year"].astype("Int64")
        panel = panel.merge(gdelt_agg, on=["iso3", "year"], how="left")
        print(f"GDELT: {len(gdelt_agg.columns)-2} annual aggregate columns added")

# ── FAO Food Price Index → annual mean ───────────────────────────────────────
raw_ffpi = read_latest_parquet(RAW_PREFIXES["fao_ffpi"])
if raw_ffpi is not None:
    raw_ffpi.columns = [c.lower().strip() for c in raw_ffpi.columns]
    # FFPI is a global series (no country column) — broadcast to all countries
    val_col  = next((c for c in raw_ffpi.columns if "index_value" in c or c == "value"), None)
    name_col = next((c for c in raw_ffpi.columns if "index_name" in c or c == "name"), None)
    yr_col   = next((c for c in raw_ffpi.columns if c == "year"), None)

    if val_col and name_col and yr_col:
        ffpi_wide = (
            raw_ffpi.groupby([yr_col, name_col])[val_col].mean()
            .unstack(name_col)
            .reset_index()
            .rename(columns={yr_col: "year"})
        )
        ffpi_wide.columns = ["year"] + [
            f"ffpi_{c.lower().replace(' ','_')}" for c in ffpi_wide.columns[1:]
        ]
        ffpi_wide["year"] = pd.to_numeric(ffpi_wide["year"], errors="coerce").astype("Int64")
        # Broadcast: cross-join with all iso3s via spine year
        panel = panel.merge(ffpi_wide, on="year", how="left")
        print(f"FAO FFPI: {len(ffpi_wide.columns)-1} global price index columns added")

print(f"\nPanel after monthly sources: {panel.shape}")